# HW03
学号：20234080324  
姓名：李英华

## 2 卷积和池化层

### 2.1 理论计算题

卷积输出空间尺寸公式为：

$$H_{out}=W_{out}=\left\lfloor\frac{N+2P-K}{S}\right\rfloor+1$$

其中 $N=32$，$P=2$，$K=5$，$S=2$：

$$H_{out}=W_{out}=\left\lfloor\frac{32+2\times2-5}{2}\right\rfloor+1
=\lfloor 15.5\rfloor+1=16$$

卷积核个数为 16，因此输出特征图尺寸为：

$$16\times16\times16$$

单个输出通道的一个像素值需要和一个大小为 $3\times5\times5$ 的卷积核做点乘，所以乘法次数为：

$$3\times5\times5=75$$

In [1]:
import numpy as np


def max_pool2d(x, kernel_size, stride=None, padding=0):
    """手动实现 2D Max Pooling，支持 2D、3D(C,H,W)、4D(N,C,H,W) 输入。"""
    x = np.asarray(x, dtype=float)
    original_ndim = x.ndim

    if original_ndim == 2:
        x = x[None, None, :, :]
    elif original_ndim == 3:
        x = x[None, :, :, :]
    elif original_ndim != 4:
        raise ValueError("x 必须是 2D、3D 或 4D 数组")

    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size

    if stride is None:
        stride = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride

    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding

    x_pad = np.pad(
        x,
        pad_width=((0, 0), (0, 0), (ph, ph), (pw, pw)),
        mode="constant",
        constant_values=-np.inf,
    )

    n, c, h, w = x_pad.shape
    out_h = (h - kh) // sh + 1
    out_w = (w - kw) // sw + 1
    y = np.empty((n, c, out_h, out_w), dtype=x.dtype)

    for i in range(out_h):
        for j in range(out_w):
            window = x_pad[:, :, i * sh : i * sh + kh, j * sw : j * sw + kw]
            y[:, :, i, j] = window.max(axis=(2, 3))

    if original_ndim == 2:
        return y[0, 0]
    if original_ndim == 3:
        return y[0]
    return y


X = np.arange(1, 17, dtype=float).reshape(1, 1, 4, 4)
print("输入 X:")
print(X)
print("\n2x2 最大池化, stride=2:")
print(max_pool2d(X, kernel_size=2, stride=2, padding=0))
print("\n2x2 最大池化, stride=1, padding=1:")
print(max_pool2d(X[0, 0], kernel_size=2, stride=1, padding=1))

输入 X:
[[[[ 1.  2.  3.  4.]
   [ 5.  6.  7.  8.]
   [ 9. 10. 11. 12.]
   [13. 14. 15. 16.]]]]

2x2 最大池化, stride=2:
[[[[ 6.  8.]
   [14. 16.]]]]

2x2 最大池化, stride=1, padding=1:
[[ 1.  2.  3.  4.  4.]
 [ 5.  6.  7.  8.  8.]
 [ 9. 10. 11. 12. 12.]
 [13. 14. 15. 16. 16.]
 [13. 14. 15. 16. 16.]]


## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

输入和输出通道数均为 $C$。

1. 一个 $5\times5$ 卷积层（不带偏置）的参数量为：

$$C\times C\times5\times5=25C^2$$

2. 两个串联的 $3\times3$ 卷积层（不带偏置，每层通道数都为 $C$）的总参数量为：

$$2\times C\times C\times3\times3=18C^2$$

因此，两个 $3\times3$ 卷积既能获得等效 $5\times5$ 的感受野，又比单个 $5\times5$ 卷积少 $7C^2$ 个参数，并且中间多了一次非线性激活。

In [2]:
import torch
from torch import nn


def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )


block = nin_block(3, 16, kernel_size=3, stride=1, padding=1)
print(block)

Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)


## 4 Inception, 批量归一化和残差网络

### 4.1 理论计算题

给定 $x_1=2,x_2=4,x_3=6,x_4=8$。

批量均值为：

$$\mu=\frac{2+4+6+8}{4}=5$$

批量方差为：

$$\sigma^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}=5$$

标准化结果为：

$$\hat{x_i}=\frac{x_i-5}{\sqrt{5}}$$

因为 $\gamma=2,\beta=1,\epsilon=0$，所以：

$$y_i=2\hat{x_i}+1$$

最终结果：

$$y_1=1-\frac{6}{\sqrt5}\approx -1.6833$$

$$y_2=1-\frac{2}{\sqrt5}\approx 0.1056$$

$$y_3=1+\frac{2}{\sqrt5}\approx 1.8944$$

$$y_4=1+\frac{6}{\sqrt5}\approx 3.6833$$

In [3]:
import torch
from torch import nn
import torch.nn.functional as F


class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, padding=1, stride=strides
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if use_1x1conv:
            self.conv3 = nn.Conv2d(
                in_channels, out_channels, kernel_size=1, stride=strides
            )
        else:
            self.conv3 = None

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3 is not None:
            x = self.conv3(x)
        y = y + x
        return F.relu(y)


X = torch.rand(size=(4, 3, 6, 6))
blk_same = Residual(3, 3)
print("不改变形状:", blk_same(X).shape)

blk_down = Residual(3, 6, use_1x1conv=True, strides=2)
print("使用 1x1 卷积并 stride=2:", blk_down(X).shape)

不改变形状: torch.Size([4, 3, 6, 6])
使用 1x1 卷积并 stride=2: torch.Size([4, 6, 3, 3])


## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

1. 预训练网络的底层特征提取层通常已经学到了较通用的视觉特征，例如边缘、纹理、局部形状等。新任务的数据集往往更小，如果对这些层使用过大的学习率，容易破坏已经学好的通用特征并导致过拟合。因此通常将底层特征提取层冻结，或使用较小学习率微调。最终输出层是针对新任务类别数重新初始化的，原来的分类权重不能直接适配新数据集，所以需要较大的学习率来更快学习新的类别映射。

2. 如果目标数据集非常小，且与源数据集非常相似，适合把预训练模型作为固定特征提取器：冻结大部分甚至全部底层卷积层，只替换并训练最后的分类输出层；如果还需要微调，可以只解冻最后少数几层，并使用很小的学习率。同时配合数据增广、权重衰减和早停来降低过拟合风险。

In [4]:
from torchvision import transforms


train_transforms = transforms.Compose(
    [
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        transforms.ToTensor(),
    ]
)

print("数据增广管道已创建，共", len(train_transforms.transforms), "步")
for i, transform in enumerate(train_transforms.transforms, 1):
    print(f"{i}. {transform.__class__.__name__}")

数据增广管道已创建，共 4 步
1. RandomResizedCrop
2. RandomHorizontalFlip
3. ColorJitter
4. ToTensor


## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

边界框：

$$A=[10,10,50,50],\quad B=[30,30,70,70]$$

交集左上角坐标为：

$$[\max(10,30),\max(10,30)]=[30,30]$$

交集右下角坐标为：

$$[\min(50,70),\min(50,70)]=[50,50]$$

因此交集面积：

$$S_{inter}=(50-30)\times(50-30)=400$$

两个框面积分别为：

$$S_A=(50-10)\times(50-10)=1600$$

$$S_B=(70-30)\times(70-30)=1600$$

并集面积：

$$S_{union}=1600+1600-400=2800$$

所以：

$$IoU=\frac{400}{2800}=\frac{1}{7}\approx0.142857$$

In [5]:
import torch
import torch.nn.functional as F


def label_smoothing_cross_entropy(
    logits, targets, epsilon=0.1, reduction="mean", return_smoothed=False
):
    """计算标签平滑后的交叉熵损失。

    logits: 形状为 (batch_size, num_classes) 的模型原始输出
    targets: 形状为 (batch_size,) 的真实类别下标
    """
    num_classes = logits.size(1)
    log_probs = F.log_softmax(logits, dim=1)

    with torch.no_grad():
        smoothed_targets = torch.full_like(log_probs, epsilon / (num_classes - 1))
        smoothed_targets.scatter_(1, targets.unsqueeze(1), 1 - epsilon)

    loss = -(smoothed_targets * log_probs).sum(dim=1)

    if reduction == "mean":
        loss_value = loss.mean()
    elif reduction == "sum":
        loss_value = loss.sum()
    elif reduction == "none":
        loss_value = loss
    else:
        raise ValueError("reduction 必须是 'mean'、'sum' 或 'none'")

    if return_smoothed:
        return loss_value, smoothed_targets
    return loss_value


logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.2, 0.3]])
targets = torch.tensor([0, 1])

loss_each, smoothed = label_smoothing_cross_entropy(
    logits, targets, epsilon=0.1, reduction="none", return_smoothed=True
)
loss_mean = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)

print("平滑后的目标分布:")
print(smoothed)
print("每个样本的损失:")
print(loss_each)
print("平均损失:")
print(loss_mean)

平滑后的目标分布:
tensor([[0.9000, 0.0500, 0.0500],
        [0.0500, 0.9000, 0.0500]])
每个样本的损失:
tensor([0.4663, 0.6536])
平均损失:
tensor(0.5599)
